In [4]:
import ee

In [3]:
import geemap

In [5]:
ee.Authenticate()

In [6]:
ee.Initialize(project= "bhakti-ee")

In [7]:
map = geemap.Map()
map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(childr…

In [8]:
district = ee.FeatureCollection("projects/bhakti-ee/assets/DISTRICT_BOUNDARY")

In [9]:
map.addLayer(district, {}, "District")
map

Map(bottom=812.0, center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=Search…

In [10]:
MOD44B = ee.ImageCollection("MODIS/006/MOD44B").select("Percent_Tree_Cover")

In [11]:
tree_cover2002 = MOD44B.filterDate('2002-01-01', '2002-12-31').mean().clip(district)

tree_cover2020 = MOD44B.filterDate('2020-01-01', '2020-12-31').mean().clip(district)

vis_tree2002 = {'min': 0,'max': 77,'palette': ['#f7f4f0', '#e8dcc8', '#d4c19a', '#c8e6a0', '#a8d96e', '#7cc242', '#5aaa1c', '#3d8b14', '#2d6b0f', '#1a4d0a']}
vis_tree2020 = {'min': 0,'max': 77,'palette': ['#f7f4f0', '#e8dcc8', '#d4c19a', '#c8e6a0', '#a8d96e', '#7cc242', '#5aaa1c', '#3d8b14', '#2d6b0f', '#1a4d0a']}
map.addLayer(tree_cover2002, vis_tree2002, "tree_cover2002")
map.addLayer(tree_cover2020, vis_tree2020, "tree_cover2020")
map

Map(bottom=320.0, center=[81.92318632602199, 229.21981942424546], controls=(WidgetControl(options=['position',…

<IPython.core.display.Javascript object>

vis_params = {'bands': ['Percent_Tree_Cover'], 'palette': ['#f7f4f0', '#e8dcc8', '#d4c19a', '#c8e6a0', '#a8d96e', '#7cc242', '#5aaa1c', '#3d8b14', '#2d6b0f', '#1a4d0a'], 'min': 0.0, 'max': 74.0}


<IPython.core.display.Javascript object>

vis_params = {'bands': ['Percent_Tree_Cover'], 'palette': ['#f7f4f0', '#e8dcc8', '#d4c19a', '#c8e6a0', '#a8d96e', '#7cc242', '#5aaa1c', '#3d8b14', '#2d6b0f', '#1a4d0a'], 'min': 0.0, 'max': 75.0}


In [12]:
def compute_zonal_stats(image, features, scale=500):
    """
    Compute zonal statistics for each district
    """
    def calc_stats(feature):
        clipped = image.clip(feature.geometry())
        stats = clipped.reduceRegion(
            reducer=ee.Reducer.mean().combine(
                ee.Reducer.min(), '', True).combine(
                ee.Reducer.max(), '', True).combine(
                ee.Reducer.stdDev(), '', True),
            geometry=feature.geometry(),
            scale=scale,
            maxPixels=1e13
        )
        return feature.set(stats)

    return features.map(calc_stats)


district_tree2002 = compute_zonal_stats(tree_cover2002, district, scale=250)
district_tree2020 = compute_zonal_stats(tree_cover2020, district, scale=250)

In [13]:
import geemap

geemap.ee_to_geojson(district_tree2002, filename='MOD44B_TreeCover2002.geojson')
print("Exported: MOD44B_TreeCover2002.geojson")

geemap.ee_to_geojson(district_tree2020, filename='MOD44B_TreeCover2020.geojson')
print("Exported: MOD44B_TreeCover2020.geojson")

Exported: MOD44B_TreeCover2002.geojson
Exported: MOD44B_TreeCover2020.geojson


In [14]:
MCD15A3H = ee.ImageCollection("MODIS/061/MCD15A3H").select("Lai")

In [15]:
lai2005 = MCD15A3H.filterDate('2005-01-01', '2005-12-31').median().multiply(0.1).clip(district)
lai2023 = MCD15A3H.filterDate('2023-01-01', '2023-12-31').median().multiply(0.1).clip(district)
vis_lai2005 = {'min': 0, 'max': 5.3, 'palette': ['#f7fcf5', '#e7f6e2', '#ceecc7', '#aedea7', '#88cd86', '#5db96b', '#37a055', '#1b843f', '#00682a', '#00441b']}
vis_lai2023 = {'min': 0, 'max': 5.3, 'palette': ['#f7fcf5', '#e7f6e2', '#ceecc7', '#aedea7', '#88cd86', '#5db96b', '#37a055', '#1b843f', '#00682a', '#00441b']}
map.addLayer(lai2005, vis_lai2005, "lai2005")
map.addLayer(lai2023, vis_lai2023, "lai2023")
map

Map(bottom=1070.0, center=[40.713955826286046, 77.51953125000001], controls=(WidgetControl(options=['position'…

<IPython.core.display.Javascript object>

vis_params = {'bands': ['Lai'], 'palette': ['#f7fcf5', '#e7f6e2', '#ceecc7', '#aedea7', '#88cd86', '#5db96b', '#37a055', '#1b843f', '#00682a', '#00441b'], 'min': 0.0, 'max': 5.0}


<IPython.core.display.Javascript object>

vis_params = {'bands': ['Lai'], 'palette': ['#f7fcf5', '#e7f6e2', '#ceecc7', '#aedea7', '#88cd86', '#5db96b', '#37a055', '#1b843f', '#00682a', '#00441b'], 'min': 0.0, 'max': 5.0}


In [16]:
def compute_zonal_stats(image, features, scale=500):
    """
    Compute zonal statistics for each district
    """
    def calc_stats(feature):
        clipped = image.clip(feature.geometry())
        stats = clipped.reduceRegion(
            reducer=ee.Reducer.mean().combine(
                ee.Reducer.min(), '', True).combine(
                ee.Reducer.max(), '', True).combine(
                ee.Reducer.stdDev(), '', True),
            geometry=feature.geometry(),
            scale=scale,
            maxPixels=1e13
        )
        return feature.set(stats)

    return features.map(calc_stats)

district_lai2005 = compute_zonal_stats(lai2005, district, scale=500)
district_lai2023 = compute_zonal_stats(lai2023, district, scale=500)

In [17]:
import geemap

geemap.ee_to_geojson(district_lai2005, filename='MCD15A3H_LAI2005.geojson')
print("Exported: MCD15A3H_LAI2005.geojson")

geemap.ee_to_geojson(district_lai2023, filename='MCD15A3H_LAI2023.geojson')
print("Exported: MCD15A3H_LAI2023.geojson")

Exported: MCD15A3H_LAI2005.geojson
Exported: MCD15A3H_LAI2023.geojson


In [18]:
MOD11A1 = ee.ImageCollection("MODIS/061/MOD11A1").select("LST_Day_1km")

In [19]:
lst_2006 = MOD11A1.filterDate('2006-01-01', '2006-12-31').mean().multiply(0.02).subtract(273.15).clip(district)

lst_2022 = MOD11A1.filterDate('2022-01-01', '2022-12-31').mean().multiply(0.02).subtract(273.15).clip(district)

vis_lst06 = {'min': 15, 'max': 45, 'palette': ['#2166ac', '#4393c3', '#92c5de', '#d1e5f0', '#fef8b8', '#fee08b', '#fdae61', '#f46d43', '#d73027', '#a50026']}
vis_lst22 = {'min': 15, 'max': 45, 'palette': ['#2166ac', '#4393c3', '#92c5de', '#d1e5f0', '#fef8b8', '#fee08b', '#fdae61', '#f46d43', '#d73027', '#a50026']}
map.addLayer(lst_2006, vis_lst06, "lst_2006")
map.addLayer(lst_2022, vis_lst22, "lst_2022")
map

Map(bottom=1070.0, center=[40.713955826286046, 77.51953125000001], controls=(WidgetControl(options=['position'…

In [20]:
def compute_zonal_stats(image, features, scale=500):
    """
    Compute zonal statistics for each district
    """
    def calc_stats(feature):
        clipped = image.clip(feature.geometry())
        stats = clipped.reduceRegion(
            reducer=ee.Reducer.mean().combine(
                ee.Reducer.min(), '', True).combine(
                ee.Reducer.max(), '', True).combine(
                ee.Reducer.stdDev(), '', True),
            geometry=feature.geometry(),
            scale=scale,
            maxPixels=1e13
        )
        return feature.set(stats)

    return features.map(calc_stats)

# Compute zonal statistics for LST
district_lst2006 = compute_zonal_stats(lst_2006, district, scale=1000)
district_lst2022 = compute_zonal_stats(lst_2022, district, scale=1000)

In [21]:
import geemap

geemap.ee_to_geojson(district_lst2006, filename='MOD11A1_LST2006.geojson')
print("Exported: MOD11A1_LST2006.geojson")

geemap.ee_to_geojson(district_lst2022, filename='MOD11A1_LST2022.geojson')
print("Exported: MOD11A1_LST2022.geojson")

Exported: MOD11A1_LST2006.geojson
Exported: MOD11A1_LST2022.geojson
